In [2]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import zscore

In [3]:
df = pd.read_csv('academic_meals_elementary_district.csv')

In [8]:
ses_cols = ['student_low_income_percent', 'student_disabilities_percent']
hei_total_col = ['HEI 2015 Total Score']
hei_components_cols = [
    'HEI 2015 Total Fruits (0-5)', 'HEI 2015 Total Vegetables (0-5)',
    'HEI 2015 Whole Grains (0-10)', 'HEI 2015 Dairy (0-10)',
    'HEI 2015 Total Protein Foods (0-5)', 'HEI 2015 Seafood and Plant Proteins (0-5)',
    'HEI 2015 Fatty Acids (0-10)', 'HEI 2015 Refined Grains (0-10)',
    'HEI 2015 Sodium (0-10)', 'HEI 2015 Added Sugars (0-10)', 'HEI 2015 Saturated Fats (0-10)'
]
targets = ['ELA_Proficiency', 'Math_Proficiency', 'Science_Proficiency']
df_clean = df[ses_cols + hei_total_col + hei_components_cols + targets].dropna()

df_beta = df_clean.apply(zscore)

In [28]:
def check_vif(X):
    vif_df = pd.DataFrame()
    vif_df["Feature"] = X.columns
    vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_df

In [51]:
print("Model 1: HEI Total + HEI Components")
X1_vars = hei_total_col + hei_components_cols

for target in targets:
    X1 = sm.add_constant(df_clean[X1_vars])
    m1 = sm.OLS(df_clean[target], X1).fit()
    print(f"{target:20} | R²: {m1.rsquared:.4f} | Total Score Coef: {m1.params[hei_total_col[0]]:.4f} | p-val: {m1.pvalues[hei_total_col[0]]:.4f}")

Model 1: HEI Total + HEI Components
ELA_Proficiency      | R²: 0.3725 | Total Score Coef: 0.8743 | p-val: 0.8158
Math_Proficiency     | R²: 0.3474 | Total Score Coef: 2.7444 | p-val: 0.4762
Science_Proficiency  | R²: 0.3231 | Total Score Coef: 0.2981 | p-val: 0.9352


In [54]:
print("\nVIF Diagnostic for Model 1")
print(check_vif(sm.add_constant(df_clean[X1_vars])))


VIF Diagnostic for Model 1
                                      Feature         VIF
0                                       const  297.306317
1                        HEI 2015 Total Score  158.649124
2                 HEI 2015 Total Fruits (0-5)   20.585234
3             HEI 2015 Total Vegetables (0-5)   11.422033
4                HEI 2015 Whole Grains (0-10)   13.693835
5                       HEI 2015 Dairy (0-10)   16.027099
6          HEI 2015 Total Protein Foods (0-5)    4.617368
7   HEI 2015 Seafood and Plant Proteins (0-5)    9.585226
8                 HEI 2015 Fatty Acids (0-10)    7.849224
9              HEI 2015 Refined Grains (0-10)   10.898473
10                     HEI 2015 Sodium (0-10)   10.429614
11               HEI 2015 Added Sugars (0-10)    6.955268
12             HEI 2015 Saturated Fats (0-10)   12.411791


In [59]:
print("\nModel 2: HEI Total Score Only")
X2 = sm.add_constant(df_clean[hei_total_col])

for target in targets:
    m2 = sm.OLS(df_clean[target], X2).fit()
    print(f"{target:20} | R²: {m2.rsquared:.4f} | Coef: {m2.params[hei_total_col[0]]:.4f} | p-val: {m2.pvalues[hei_total_col[0]]:.4f}")


Model 2: HEI Total Score Only
ELA_Proficiency      | R²: 0.1267 | Coef: -0.7390 | p-val: 0.0177
Math_Proficiency     | R²: 0.0808 | Coef: -0.5920 | p-val: 0.0614
Science_Proficiency  | R²: 0.0682 | Coef: -0.5101 | p-val: 0.0869


In [58]:
print("Model 3: HEI Components Only")
X3_vars = hei_components_cols

for target in targets:
    X3 = sm.add_constant(df_clean[X3_vars])
    m3 = sm.OLS(df_clean[target], X3).fit()
    print(f"{target:20} | R²: {m3.rsquared:.4f} | Adj. R²: {m3.rsquared_adj:.4f}")

Model 3: HEI Components Only
ELA_Proficiency      | R²: 0.3714 | Adj. R²: 0.1554
Math_Proficiency     | R²: 0.3364 | Adj. R²: 0.1083
Science_Proficiency  | R²: 0.3229 | Adj. R²: 0.0902


In [60]:
print("\nVIF Diagnostic for Model 3")
print(check_vif(sm.add_constant(df_clean[X3_vars])))


VIF Diagnostic for Model 3
                                      Feature         VIF
0                                       const  295.967837
1                 HEI 2015 Total Fruits (0-5)    2.615066
2             HEI 2015 Total Vegetables (0-5)    2.244686
3                HEI 2015 Whole Grains (0-10)    2.382153
4                       HEI 2015 Dairy (0-10)    3.066023
5          HEI 2015 Total Protein Foods (0-5)    2.817880
6   HEI 2015 Seafood and Plant Proteins (0-5)    3.462965
7                 HEI 2015 Fatty Acids (0-10)    3.443677
8              HEI 2015 Refined Grains (0-10)    3.752252
9                      HEI 2015 Sodium (0-10)    5.201813
10               HEI 2015 Added Sugars (0-10)    4.103160
11             HEI 2015 Saturated Fats (0-10)    4.328629


In [31]:
print("Model 4: SES + Components for ELA")
target_ela = 'ELA_Proficiency'
print(f"\n{'='*30} {target_ela} MODEL 4 {'='*30}")

X4_ela = sm.add_constant(df_clean[ses_cols + hei_components_cols])
model4_ela = sm.OLS(df_clean[target_ela], X4_ela).fit()
print(model4_ela.summary())


Model 4: SES + Components for ELA

============================== ELA_Proficiency MODEL 4 ==============================
                            OLS Regression Results                            
Dep. Variable:        ELA_Proficiency   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.608
Method:                 Least Squares   F-statistic:                     6.130
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           2.08e-05
Time:                        11:58:12   Log-Likelihood:                -162.11
No. Observations:                  44   AIC:                             352.2
Df Residuals:                      30   BIC:                             377.2
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|    

In [30]:
print("\nVIF Check for ELA Model")
print(check_vif(X4_ela))


VIF Check for ELA Model
                                      Feature         VIF
0                                       const  333.696668
1                  student_low_income_percent    1.338360
2                student_disabilities_percent    1.140514
3                 HEI 2015 Total Fruits (0-5)    2.715118
4             HEI 2015 Total Vegetables (0-5)    2.353889
5                HEI 2015 Whole Grains (0-10)    2.523481
6                       HEI 2015 Dairy (0-10)    3.135171
7          HEI 2015 Total Protein Foods (0-5)    2.960328
8   HEI 2015 Seafood and Plant Proteins (0-5)    3.534259
9                 HEI 2015 Fatty Acids (0-10)    3.534774
10             HEI 2015 Refined Grains (0-10)    3.878684
11                     HEI 2015 Sodium (0-10)    5.215052
12               HEI 2015 Added Sugars (0-10)    4.177791
13             HEI 2015 Saturated Fats (0-10)    4.369491


In [47]:
X_beta_ela = sm.add_constant(df_beta[ses_cols + hei_components_cols])
model_beta_ela = sm.OLS(df_beta[target_ela], X_beta_ela).fit()

print(f"\n{'='*20} ELA: Top 5 Predictors for ELA {'='*20}")
ela_results = pd.DataFrame({
    'Beta': model_beta_ela.params,
    'P-Value': model_beta_ela.pvalues
}).drop('const')

ela_results['Abs_Beta'] = ela_results['Beta'].abs()
top_5_ela = ela_results.sort_values(by='Abs_Beta', ascending=False).head(5)

for i, (idx, row) in enumerate(top_5_ela.iterrows()):
    sig_marker = "***" if row['P-Value'] < 0.05 else ""
    print(f"{i+1}. {idx:40} | Beta: {row['Beta']:.4f} | p-val: {row['P-Value']:.4f} {sig_marker}")


==================== ELA: Top 5 Predictors for ELA ====================
1. student_low_income_percent               | Beta: -0.6553 | p-val: 0.0000 ***
2. HEI 2015 Saturated Fats (0-10)           | Beta: 0.3484 | p-val: 0.0911 
3. HEI 2015 Refined Grains (0-10)           | Beta: -0.2612 | p-val: 0.1750 
4. HEI 2015 Total Protein Foods (0-5)       | Beta: -0.2560 | p-val: 0.1296 
5. HEI 2015 Total Fruits (0-5)              | Beta: -0.2516 | p-val: 0.1203 


In [33]:
print("Model 4: SES + Components for Math")
target_math = 'Math_Proficiency'
print(f"\n{'='*30} {target_math} MODEL 4 {'='*30}")

X4_math = sm.add_constant(df_clean[ses_cols + hei_components_cols])
model4_math = sm.OLS(df_clean[target_math], X4_math).fit()

print(model4_math.summary())

Model 4: SES + Components for Math

============================== Math_Proficiency MODEL 4 ==============================
                            OLS Regression Results                            
Dep. Variable:       Math_Proficiency   R-squared:                       0.773
Model:                            OLS   Adj. R-squared:                  0.675
Method:                 Least Squares   F-statistic:                     7.857
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           1.74e-06
Time:                        12:04:01   Log-Likelihood:                -158.14
No. Observations:                  44   AIC:                             344.3
Df Residuals:                      30   BIC:                             369.3
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|  

In [34]:
print("\nVIF Check for Math Model")
print(check_vif(X4_math))

X_beta_math = sm.add_constant(df_beta[ses_cols + hei_components_cols])
model_beta_math = sm.OLS(df_beta[target_math], X_beta_math).fit()



VIF Check for Math Model
                                      Feature         VIF
0                                       const  333.696668
1                  student_low_income_percent    1.338360
2                student_disabilities_percent    1.140514
3                 HEI 2015 Total Fruits (0-5)    2.715118
4             HEI 2015 Total Vegetables (0-5)    2.353889
5                HEI 2015 Whole Grains (0-10)    2.523481
6                       HEI 2015 Dairy (0-10)    3.135171
7          HEI 2015 Total Protein Foods (0-5)    2.960328
8   HEI 2015 Seafood and Plant Proteins (0-5)    3.534259
9                 HEI 2015 Fatty Acids (0-10)    3.534774
10             HEI 2015 Refined Grains (0-10)    3.878684
11                     HEI 2015 Sodium (0-10)    5.215052
12               HEI 2015 Added Sugars (0-10)    4.177791
13             HEI 2015 Saturated Fats (0-10)    4.369491


In [46]:
X_beta_math = sm.add_constant(df_beta[ses_cols + hei_components_cols])
model_beta_math = sm.OLS(df_beta[target_math], X_beta_math).fit()

print(f"\n{'='*20} Math: Top 5 Predictors for Math {'='*20}")
math_results = pd.DataFrame({
    'Beta': model_beta_math.params,
    'P-Value': model_beta_math.pvalues
}).drop('const')

math_results['Abs_Beta'] = math_results['Beta'].abs()
top_5_math = math_results.sort_values(by='Abs_Beta', ascending=False).head(5)

for i, (idx, row) in enumerate(top_5_math.iterrows()):
    sig_marker = "***" if row['P-Value'] < 0.05 else ""
    print(f"{i+1}. {idx:40} | Beta: {row['Beta']:.4f} | p-val: {row['P-Value']:.4f} {sig_marker}")


==================== Math: Top 5 Predictors for Math ====================
1. student_low_income_percent               | Beta: -0.7316 | p-val: 0.0000 ***
2. HEI 2015 Refined Grains (0-10)           | Beta: -0.3114 | p-val: 0.0792 
3. HEI 2015 Saturated Fats (0-10)           | Beta: 0.2386 | p-val: 0.1995 
4. HEI 2015 Whole Grains (0-10)             | Beta: -0.2214 | p-val: 0.1197 
5. HEI 2015 Total Fruits (0-5)              | Beta: -0.2113 | p-val: 0.1508 


In [36]:
print("Model 4: SES + Components for Science")
target_sci = 'Science_Proficiency'
print(f"\n{'='*30} {target_sci} MODEL 4 {'='*30}")

X4_sci = sm.add_constant(df_clean[ses_cols + hei_components_cols])
model4_sci = sm.OLS(df_clean[target_sci], X4_sci).fit()

print(model4_sci.summary())

Model 4: SES + Components for Science

============================== Science_Proficiency MODEL 4 ==============================
                             OLS Regression Results                            
Dep. Variable:     Science_Proficiency   R-squared:                       0.721
Model:                             OLS   Adj. R-squared:                  0.600
Method:                  Least Squares   F-statistic:                     5.969
Date:                 Tue, 24 Mar 2026   Prob (F-statistic):           2.68e-05
Time:                         12:05:16   Log-Likelihood:                -159.85
No. Observations:                   44   AIC:                             347.7
Df Residuals:                       30   BIC:                             372.7
Df Model:                           13                                         
Covariance Type:             nonrobust                                         
                                                coef    std err        

In [37]:
print("\nVIF Check for Science Model")
print(check_vif(X4_sci))


VIF Check for Science Model
                                      Feature         VIF
0                                       const  333.696668
1                  student_low_income_percent    1.338360
2                student_disabilities_percent    1.140514
3                 HEI 2015 Total Fruits (0-5)    2.715118
4             HEI 2015 Total Vegetables (0-5)    2.353889
5                HEI 2015 Whole Grains (0-10)    2.523481
6                       HEI 2015 Dairy (0-10)    3.135171
7          HEI 2015 Total Protein Foods (0-5)    2.960328
8   HEI 2015 Seafood and Plant Proteins (0-5)    3.534259
9                 HEI 2015 Fatty Acids (0-10)    3.534774
10             HEI 2015 Refined Grains (0-10)    3.878684
11                     HEI 2015 Sodium (0-10)    5.215052
12               HEI 2015 Added Sugars (0-10)    4.177791
13             HEI 2015 Saturated Fats (0-10)    4.369491


In [45]:
X_beta_sci = sm.add_constant(df_beta[ses_cols + hei_components_cols])
model_beta_sci = sm.OLS(df_beta[target_sci], X_beta_sci).fit()

print(f"\n{'='*20} Science: Top 5 Predictors for Science {'='*20}")
sci_results = pd.DataFrame({
    'Beta': model_beta_sci.params,
    'P-Value': model_beta_sci.pvalues
}).drop('const')

sci_results['Abs_Beta'] = sci_results['Beta'].abs()
top_5_sci = sci_results.sort_values(by='Abs_Beta', ascending=False).head(5)

for i, (idx, row) in enumerate(top_5_sci.iterrows()):
    sig_marker = "***" if row['P-Value'] < 0.05 else ""
    print(f"{i+1}. {idx:40} | Beta: {row['Beta']:.4f} | p-val: {row['P-Value']:.4f} {sig_marker}")


==================== Science: Top 5 Predictors for Science ====================
1. student_low_income_percent               | Beta: -0.6261 | p-val: 0.0000 ***
2. HEI 2015 Whole Grains (0-10)             | Beta: -0.3241 | p-val: 0.0427 ***
3. HEI 2015 Saturated Fats (0-10)           | Beta: 0.2899 | p-val: 0.1607 
4. HEI 2015 Total Protein Foods (0-5)       | Beta: -0.2853 | p-val: 0.0958 
5. HEI 2015 Total Fruits (0-5)              | Beta: -0.2376 | p-val: 0.1452 


Findings from Beta Estimates

SES Dominance: Low-income status (student_low_income_percent) is the strongest predictor across all subjects, with the most severe impact on Math (Beta: -0.7316).

Saturated Fats as a Primary Driver: Lower intake of saturated fats (higher HEI score) is the top nutritional predictor for both ELA (Beta: 0.3484) and Science (Beta: 0.2899).

Subject-Specific Sensitivity: Math is uniquely sensitive to Refined Grain quality (Beta: -0.3114). Science is uniquely sensitive to Whole Grain quality (Beta: -0.3241).
